# PaliGemma Test - Experiment 2

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gomez Soteres  
**Universidad Francisco de Vitoria**

---

## Objetivo

Notebook simplificado para probar PaliGemma con imagenes del campus:
- Cargar modelo PaliGemma
- Subir imagenes y probar deteccion de Building y Floor
- Ver respuestas directas del modelo

## Uso

1. Ejecuta todas las celdas en orden
2. Sube una imagen en la celda de prueba
3. Observa la respuesta de PaliGemma

## 1. Instalacion

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 accelerate
!pip install -q torch torchvision
!pip install -q Pillow
!pip install -q huggingface_hub

print("Paquetes instalados correctamente!")

## 2. Login en Hugging Face

In [ ]:
from huggingface_hub import login

# Login interactivo
login()

## 3. Cargar modelo PaliGemma

In [ ]:
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import torch
from PIL import Image
import time

# Model configuration
model_id = "google/paligemma2-3b-mix-224"

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

print(f"\nCargando modelo: {model_id}")
print("   Esto puede tardar unos minutos...")

# Load processor
processor = AutoProcessor.from_pretrained(model_id)

# Load model
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else device,
    low_cpu_mem_usage=True
)

model.eval()

print("Modelo cargado correctamente!")

# Show GPU memory usage
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\nGPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

## 4. Configurar prompt

In [ ]:
# Prompt usado en el sistema (mismo que en paligemma_api_colab.ipynb)
PROMPT = "Read the signage board in this campus image. Identify the building name and floor number displayed on the sign. Look for words like 'BUILDING' or 'FLOOR' and the associated text."

print("PROMPT CONFIGURADO:")
print("="*60)
print(PROMPT)
print("="*60)

## 5. Funcion para hacer preguntas a PaliGemma

In [ ]:
def ask_paligemma(image: Image.Image, prompt: str = None, max_tokens: int = 100) -> dict:
    """
    Hace una pregunta a PaliGemma sobre una imagen.
    
    Args:
        image: Imagen PIL
        prompt: Pregunta en texto (si None, usa PROMPT por defecto)
        max_tokens: Maximo de tokens a generar
    
    Returns:
        Dict con respuesta y tiempo de inferencia
    """
    prompt_to_use = prompt if prompt else PROMPT
    
    # Prepare inputs
    inputs = processor(
        text=prompt_to_use,
        images=image,
        padding="longest",
        do_convert_rgb=True,
        return_tensors="pt"
    ).to(model.device)
    
    inputs = inputs.to(dtype=model.dtype)
    
    # Generate response
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    inference_time = time.time() - start_time
    
    # Decode response
    input_len = inputs["input_ids"].shape[-1]
    generated_ids = outputs[0][input_len:]
    response = processor.decode(generated_ids, skip_special_tokens=True).strip()
    
    return {
        "answer": response,
        "inference_time": inference_time
    }

print("Funcion lista!")

## 6. Probar con una imagen

In [ ]:
from google.colab import files
import io

# Subir imagen
print("Sube una imagen de cartel del campus:")
uploaded = files.upload()

# Cargar imagen
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

print(f"\nImagen cargada: {filename}")
print(f"Tamano: {image.size}")

# Mostrar imagen
display(image.resize((500, 400)))

# Ejecutar pregunta
print("\n" + "="*60)
print("EJECUTANDO PREGUNTA A PALIGEMMA")
print("="*60)
print(f"Modelo: {model_id}")
print("="*60)

result = ask_paligemma(image, PROMPT)

print("\n" + "="*60)
print("RESPUESTA DE PALIGEMMA:")
print("="*60)
print(f"\n>>> {result['answer']}\n")
print("="*60)
print(f"Tiempo de inferencia: {result['inference_time']:.2f} segundos")
print("="*60)

## 7. Probar con otra imagen (opcional)

In [ ]:
# Ejecuta esta celda para probar con otra imagen sin recargar el modelo

from google.colab import files
import io

print("Sube otra imagen:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

print(f"\nImagen: {filename}")
display(image.resize((500, 400)))

# Ejecutar
result = ask_paligemma(image, PROMPT)

print("\n" + "="*60)
print("RESPUESTA DE PALIGEMMA:")
print("="*60)
print(f"\n>>> {result['answer']}\n")
print(f"Tiempo: {result['inference_time']:.2f}s")
print("="*60)

## 8. Probar con prompt personalizado (opcional)

In [ ]:
# Prueba con un prompt diferente si quieres

CUSTOM_PROMPT = """Look at this campus building sign carefully.
Extract the building code (EPS, CD, H) and floor number (-1, 0, 1, 2).
Answer as: Building: [CODE], Floor: [NUMBER]"""

print("PROMPT PERSONALIZADO:")
print(CUSTOM_PROMPT)
print("\n" + "="*60)

# Usar la imagen anterior o subir una nueva
# Si quieres usar la imagen anterior, simplemente ejecuta:
result_custom = ask_paligemma(image, CUSTOM_PROMPT)

print("RESPUESTA:")
print(f">>> {result_custom['answer']}")
print(f"Tiempo: {result_custom['inference_time']:.2f}s")
print("="*60)